# 文本数据与模式提取

学习目标：批量清理文本，选择字面或正则匹配，拆分和提取字段，并保留缺失与不匹配记录供核查。

前置知识：Python 字符串、正则表达式基础、缺失值。

运行环境：Python 3.12、pandas 3.0、PyArrow 25.0。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章默认 str 使用当前环境的 PyArrow 存储；需要对照时显式指定存储后端。示例输入就地构造，后续单元沿用首次导入的 pd。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 清理标识文本

收到门店编号后，先去除首尾空白，再统一成大写，保留原文本以便检查。Series 的 str 访问器把字符串方法应用到每个非缺失元素，通常保留原索引；它不是把整列拼成一个 Python 字符串。

空字符串表示已有文本但没有字符；缺失值表示没有提供文本。二者在清理后仍需区分。

In [1]:
import pandas as pd

records = pd.DataFrame(
    {"raw_code": [" ab-01 ", "Cd-02", "", None, " 北京 门店 "]},
    index=["R1", "R2", "R3", "R4", "R5"],
)
trimmed = records["raw_code"].str.strip()
records["clean_code"] = trimmed.str.upper()
print(records)  # 清理后依次为 AB-01、CD-02、空串、NaN、北京 门店。
print(records["clean_code"].isna().tolist())  # 只有 R4 为 True，空串不是缺失值。
print(records.shape, records["clean_code"].dtype)  # (5, 2) str

   raw_code clean_code
R1   ab-01       AB-01
R2    Cd-02      CD-02
R3                    
R4      NaN        NaN
R5   北京 门店       北京 门店
[False, False, False, True, False]
(5, 2) str


## 2 文本类型与缺失结果

pandas 3 默认把字符串输入推断为 str，使用 NaN 语义；显式 dtype="string" 使用 pd.NA 语义。两者都属于 StringDtype，存储后端由 dtype.storage 查看，不能仅凭显示为 string 就认定使用 Python 存储。

在当前环境中二者均使用 PyArrow。str 的长度结果遇缺失会成为 float64，string 的长度结果使用可空整数 Int64。contains 等布尔文本方法的默认缺失结果也不同；需要把缺失明确视为“不匹配”时传入 na=False。

In [2]:
default = pd.Series(["AB", "", None])
nullable = pd.Series(["AB", "", None], dtype="string")
print(default.dtype, default.dtype.storage)  # str pyarrow
print(nullable.dtype, nullable.dtype.storage)  # string pyarrow
print(default.str.len().tolist(), default.str.len().dtype)  # [2.0, 0.0, nan] float64
print(nullable.str.len().tolist(), nullable.str.len().dtype)  # [2, 0, <NA>] Int64
print(default.str.contains("A", regex=False).tolist())  # [True, False, False]
print(nullable.str.contains("A", regex=False).tolist())  # [True, False, <NA>]
print(nullable.str.contains("A", regex=False, na=False).tolist())  # [True, False, False]

str pyarrow
string pyarrow
[2.0, 0.0, nan] float64
[2, 0, <NA>] Int64
[True, False, False]
[True, False, <NA>]
[True, False, False]


## 3 空白、大小写、长度与切片

strip 默认只去除首尾空白，不删除文本中间的空格；lower 和 upper 分别转成小写与大写，不改变本例的中文字符。len 返回字符串长度，slice 按字符位置截取，起点包含、终点不包含。

下面取前两个字符。对空串切片仍得到空串；对缺失值操作仍保留缺失。

In [3]:
text = pd.Series([" Ab C ", " 北京 ", "", None], dtype="string")
clean = text.str.strip()
print(clean.tolist())  # ['Ab C', '北京', '', <NA>]
print(clean.str.lower().tolist())  # ['ab c', '北京', '', <NA>]
print(clean.str.upper().tolist())  # ['AB C', '北京', '', <NA>]
print(clean.str.len().tolist())  # [4, 2, 0, <NA>]
print(clean.str.slice(0, 2).tolist())  # ['Ab', '北京', '', <NA>]
print(clean.str[:2].equals(clean.str.slice(0, 2)))  # True：两种切片写法相同。

['Ab C', '北京', '', <NA>]


['ab c', '北京', '', <NA>]
['AB C', '北京', '', <NA>]
[4, 2, 0, <NA>]
['Ab', '北京', '', <NA>]
True


## 4 字面包含与正则包含

contains 判断文本任意位置是否包含目标。查找普通标点时使用 regex=False；regex=True 则把模式作为正则表达式。两种解释可能产生完全不同的结果。

例如点号作为字面字符只匹配句点；作为正则模式时，它通常表示一个非换行字符。case=False 可忽略大小写；na=False 是本例把缺失视为不匹配的明确约定。

In [4]:
text = pd.Series(["A.1", "AB1", "中文。", "", None])
literal = text.str.contains(".", regex=False, na=False)
pattern = text.str.contains(".", regex=True, na=False)
print(literal.tolist())  # [True, False, False, False, False]
print(pattern.tolist())  # [True, True, True, False, False]
print(text.str.contains("ab", case=False, regex=False, na=False).tolist())
# 只有 AB1 为 True；英文句点与中文句号也不是同一个字符。

[True, False, False, False, False]
[True, True, True, False, False]
[False, True, False, False, False]


## 5 包含、开头匹配与完整匹配

编号格式为两位大写英文字母、连字符和两位 ASCII 数字。下面的模式 [A-Z]{2}-[0-9]{2} 表示这一格式；花括号里的 2 是重复次数。

| 方法 | 中文名称／含义 | 适用任务 |
| --- | --- | --- |
| contains | 任意位置包含匹配 | 从备注中发现编号 |
| match | 从文本开头匹配 | 检查是否以编号开头 |
| fullmatch | 整段文本完整匹配 | 检查整个字段是否符合编号格式 |

match 和 fullmatch 的模式本身就是正则，没有 regex=False 参数。需要完整的字面字符串比较时，可以直接使用相等比较。

In [5]:
codes = pd.Series(["AB-01", "备注AB-01", "AB-01尾注", "ab-01", "", None])
pattern = r"[A-Z]{2}-[0-9]{2}"
checks = pd.DataFrame(
    {
        "text": codes,
        "contains": codes.str.contains(pattern, regex=True, na=False),
        "match": codes.str.match(pattern, na=False),
        "fullmatch": codes.str.fullmatch(pattern, na=False),
    }
)
print(checks)
# AB-01 三项都为 True；备注AB-01 仅 contains 为 True；AB-01尾注 前两项为 True。
# 小写、空串、缺失值三项均为 False。
print(checks.shape)  # (6, 4)，没有删除不匹配记录。

      text  contains  match  fullmatch
0    AB-01      True   True       True
1  备注AB-01      True  False      False
2  AB-01尾注      True   True      False
3    ab-01     False  False      False
4              False  False      False
5      NaN     False  False      False
(6, 4)


## 6 拆分地址字段

地址文本已按固定分隔符组织时，用 split 拆分比编写提取模式更直接。n 限制拆分次数，expand=True 把结果展开为列；默认 expand=False 则返回列表组成的 Series。

下面只按第一个竖线拆分，把后面的详细地址作为整体保留。regex=False 明确把竖线当作普通分隔符。没有分隔符的文本保留在首列，后续列缺失；原输入行不会消失。

In [6]:
addresses = pd.Series(
    ["北京|朝阳|望京", "上海|浦东", "广州", "", None],
    index=["R1", "R2", "R3", "R4", "R5"],
    dtype="string",
)
parts = addresses.str.split("|", n=1, expand=True, regex=False)
parts.columns = ["city", "detail"]
print(parts)  # R1 为 北京、朝阳|望京；R2 为 上海、浦东；R3 后一列缺失。
print(parts.loc["R4"].tolist(), parts.loc["R5"].tolist())  # ['', <NA>] 与 [<NA>, <NA>]
print(parts.shape, parts.index.tolist())  # (5, 2)，仍为 R1 至 R5。

    city detail
R1    北京  朝阳|望京
R2    上海     浦东
R3    广州   <NA>
R4         <NA>
R5  <NA>   <NA>
['', <NA>] [<NA>, <NA>]
(5, 2) ['R1', 'R2', 'R3', 'R4', 'R5']


分隔符有多种形式时，可以使用正则字符集合。下面的 [|/] 表示竖线或斜线中的任意一个字符。

split 的 regex 默认是 None，会根据模式长度等条件决定解释方式；正文显式指定 True 或 False，避免分隔符变化后意外改变语义。

In [7]:
addresses = pd.Series(["北京|朝阳", "上海/浦东", "无分隔符", None], dtype="string")
parts = addresses.str.split(r"[|/]", n=1, expand=True, regex=True)
print(parts)  # 前两行分别拆出城市与区；第三行第二列缺失，第四行两列缺失。
print(parts.shape)  # (4, 2)

      0     1
0    北京    朝阳
1    上海    浦东
2  无分隔符  <NA>
3  <NA>  <NA>
(4, 2)


## 7 捕获组与字段提取

extract 从每条文本的第一个匹配中提取捕获组；每个组对应一个结果列。命名组 (?P&lt;area&gt;...) 的 area 会成为列名，省略名字时使用组编号。

下面用两个组提取区域代码和编号。extract 不要求整段文本匹配，也不会自动把编号转成整数，因此能保留 01 这样的前导零。不匹配、空串和缺失输入均保留原行，提取字段缺失。

In [8]:
codes = pd.Series(
    ["AB-01", "备注CD-02", "无编号", "", None, "EF-03 GH-04"],
    index=["R1", "R2", "R3", "R4", "R5", "R6"],
    dtype="string",
)
pattern = r"(?P<area>[A-Z]{2})-(?P<number>[0-9]{2})"
fields = codes.str.extract(pattern, expand=True)
print(fields)  # R1：AB、01；R2：CD、02；R6 只取 EF、03；R3 至 R5 缺失。
print(fields.shape, fields.index.equals(codes.index))  # (6, 2) True
print(fields.dtypes)  # 本次两列均为 string，提取出的数字仍是文本。
print(codes.str.fullmatch(r"[A-Z]{2}-[0-9]{2}", na=False).tolist())
# 只有 R1 为 True；能提取出编号不代表整个原字段格式正确。

    area number
R1    AB     01
R2    CD     02
R3  <NA>   <NA>
R4  <NA>   <NA>
R5  <NA>   <NA>
R6    EF     03
(6, 2) True
area      string
number    string
dtype: object
[True, False, False, False, False, False]


只需要一个字段时，单个捕获组配合 expand=False 可返回 Series。extract 的模式至少要有一个捕获组；只写匹配规则而没有括号，无法指定要提取的组。

In [9]:
codes = pd.Series(["AB-01", "无编号", None], dtype="string")
number = codes.str.extract(r"[A-Z]{2}-([0-9]{2})", expand=False)
print(number.tolist(), number.dtype)  # ['01', <NA>, <NA>] string

# 预期 ValueError：正则表达式没有捕获组，extract 无法确定要提取的字段。
codes.str.extract(r"[A-Z]{2}-[0-9]{2}")

['01', <NA>, <NA>] string


ValueError: pattern contains no capture groups

contains 只回答是否匹配，不返回组内容。把捕获组传给它时，pandas 会发出 UserWarning，提醒可以使用 extract；如果括号只是为了组合模式，可使用不捕获的 (?:...)。

下面局部记录并检查这条预期警告，不关闭全局警告。实际字段提取仍使用 extract。

In [10]:
import warnings

codes = pd.Series(["AB-01", "CD-02", "无编号"])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    has_code = codes.str.contains(r"(AB|CD)-[0-9]{2}", regex=True, na=False)

assert len(caught) == 1 and caught[0].category is UserWarning
assert "has match groups" in str(caught[0].message)
print(caught[0].category.__name__)  # UserWarning：捕获组不会由 contains 返回。
print(has_code.tolist())  # [True, True, False]
print(codes.str.contains(r"(?:AB|CD)-[0-9]{2}", regex=True, na=False).tolist())
# [True, True, False]：不捕获的括号只负责组合备选区域代码。

UserWarning
[True, True, False]
[True, True, False]


## 8 替换文本

replace 根据 regex 参数选择字面替换或正则替换。str.replace 替换的是每条字符串里的片段，返回新结果；没有匹配的文本保持原样，缺失值继续缺失。

下面把英文句点改成连字符，再把连续空格或制表符改成一个连字符。字符串内部是否允许这种改写取决于编号规则，不应把地址等其他字段一起改写。

In [11]:
codes = pd.Series(["AB.01", "CD  02", "EF-03", "", None], dtype="string")
literal = codes.str.replace(".", "-", regex=False)
cleaned = literal.str.replace(r"[ 	]+", "-", regex=True)
print(literal.tolist())  # ['AB-01', 'CD  02', 'EF-03', '', <NA>]
print(cleaned.tolist())  # ['AB-01', 'CD-02', 'EF-03', '', <NA>]
print(codes.iloc[0])  # AB.01：原 Series 未被修改。

['AB-01', 'CD  02', 'EF-03', '', <NA>]
['AB-01', 'CD-02', 'EF-03', '', <NA>]
AB.01


## 9 正则与存储后端

文本 dtype 的缺失语义与存储后端是两项不同的选择。显式 string[python] 使用 Python 存储，string[pyarrow] 使用 Arrow 存储；二者都使用 pd.NA，但不能据此认定所有正则行为完全相同。

Arrow 的正则操作使用 RE2 规则，Python re 默认的 \d 则包含 Unicode 十进制数字。下面分别指定两种存储，比较 ASCII 数字和全角数字。只允许 ASCII 数字时，明确写 [0-9]；需要 Python re 语义时，可显式选择 Python 存储并核对模式。

复杂模式、flags 和具体方法也会影响支持条件；不要从某个简单模式成功推断所有模式都可跨后端保持相同行为。

In [12]:
values = ["123", "１２３", "中文", "", None]
arrow_text = pd.Series(values, dtype="string[pyarrow]")
python_text = pd.Series(values, dtype="string[python]")
print(arrow_text.dtype.storage, python_text.dtype.storage)  # pyarrow python
print(arrow_text.str.fullmatch(r"\d+", na=False).tolist())  # 本次 [True, False, False, False, False]
print(python_text.str.fullmatch(r"\d+", na=False).tolist())  # [True, True, False, False, False]
print(python_text.str.fullmatch(r"[0-9]+", na=False).tolist())  # [True, False, False, False, False]

pyarrow python
[True, False, False, False, False]
[True, True, False, False, False]
[True, False, False, False, False]


## 10 选学：提取多次匹配
一条备注可能包含多个编号，extractall 返回每次匹配，而不只取第一次。没有匹配的原行不会出现在结果中，因此要同时保存原输入，不能把结果行数当成原记录数。

结果的行标签是 MultiIndex，即由多个部分共同组成的标签。本例每个标签包含“原记录编号、该记录中的匹配序号”，末级名为 match，从 0 开始；reset_index 可把这两个部分转回普通列查看。

In [13]:
notes = pd.Series(["AB-01 CD-02", "无编号", None], index=["R1", "R2", "R3"], dtype="string")
matches = notes.str.extractall(r"(?P<area>[A-Z]{2})-(?P<number>[0-9]{2})")
print(matches)  # 只有 R1，match=0 对应 AB、01；match=1 对应 CD、02。
print(matches.index.tolist(), matches.shape)  # [('R1', 0), ('R1', 1)] (2, 2)
print(matches.reset_index())  # 原记录编号与 match 变为普通列，便于查看每次匹配来源。

         area number
   match            
R1 0       AB     01
   1       CD     02
[('R1', 0), ('R1', 1)] (2, 2)
  level_0  match area number
0      R1      0   AB     01
1      R1      1   CD     02


## 11 选学：拼接文本
cat 传入另一个 Series 时，按标签拼接对应文本，join="left" 保留左侧标签。任一侧缺失时，默认结果也缺失；na_rep 可指定展示用的缺失占位文字。

不传另一个对象时，cat 把整列合成一个字符串，默认跳过缺失值。应区分逐行拼接与整列汇总。

In [14]:
city = pd.Series(["北京", "上海", None], index=["R1", "R2", "R3"], dtype="string")
district = pd.Series(["浦东", "朝阳", "天河"], index=["R2", "R1", "R3"], dtype="string")
combined = city.str.cat(district, sep="/", join="left")
print(combined.tolist())  # ['北京/朝阳', '上海/浦东', <NA>]：按标签而非输入位置拼接。
print(city.str.cat(district, sep="/", join="left", na_rep="未知").tolist())
# ['北京/朝阳', '上海/浦东', '未知/天河']：占位文字不意味着补回了真实城市。
print(city.str.cat(sep="、"))  # 北京、上海：整列汇总时跳过缺失值。

['北京/朝阳', '上海/浦东', <NA>]
['北京/朝阳', '上海/浦东', '未知/天河']
北京、上海


## 12 选学：Unicode 规范化
外观相同的文字可能使用不同的 Unicode 字符序列。normalize 可统一其规范形式：NFC 做规范分解后再组合；NFKC 还处理兼容等价字符，例如本例的全角字母与数字。

NFKC 会改变一些原有表示，因此应根据业务决定是否把这些字符视为等价，并保留原文；它不等于通用的纠错或语言转换。

In [15]:
text = pd.Series(["é", "e\u0301", "ＡＢ－０１", "AB-01", None], dtype="string")
nfc = text.str.normalize("NFC")
nfkc = text.str.normalize("NFKC")
print(text.str.len().tolist())  # [1, 2, 5, 5, <NA>]：前两项外观相近，原字符序列长度不同。
print(nfc.iloc[0] == nfc.iloc[1])  # True：规范化后前两项相同。
print(nfkc.tolist())  # ['é', 'é', 'AB-01', 'AB-01', <NA>]
print(text.iloc[2])  # ＡＢ－０１：原文本保留。

[1, 2, 5, 5, <NA>]
True
['é', 'é', 'AB-01', 'AB-01', <NA>]
ＡＢ－０１


## 13 综合应用：保留异常记录

清理一批模拟门店编号与地址：编号只去首尾空白并转大写，必须完整符合两位大写字母、连字符和两位 ASCII 数字；地址按第一个竖线拆分。分别标记编号缺失、清理后空串和格式合格，不删除不合格记录。

这些规则是本例约定。例如全角编号没有先做兼容规范化，仍不符合格式；提取出字段也不代表原始输入已通过校验。

In [16]:
records = pd.DataFrame(
    {
        "raw_code": [" ab-01 ", "CD-02", "备注EF-03", "", None, "ＡＢ－０１"],
        "raw_address": ["北京|朝阳", "上海|浦东", "广州", "", None, "深圳|南山"],
    },
    index=["R1", "R2", "R3", "R4", "R5", "R6"],
)
clean = records["raw_code"].str.strip().str.upper()
records["clean_code"] = clean
records["missing"] = clean.isna()
records["empty"] = clean.eq("")
records["valid"] = clean.str.fullmatch(r"[A-Z]{2}-[0-9]{2}", na=False)
address_parts = records["raw_address"].str.split("|", n=1, expand=True, regex=False)
address_parts.columns = ["city", "detail"]
records[["city", "detail"]] = address_parts
print(records[["clean_code", "missing", "empty", "valid"]])
# R1、R2 合格；R4 为空串，R5 缺失；R3、R6 非空但格式不合格。
print(records[["city", "detail"]])  # R3 仅有城市；R4 为空城市且无详细地址；R5 两项缺失。
print(records.shape, records.index.tolist())  # (6, 8)，六条原记录及其顺序全部保留。

   clean_code  missing  empty  valid
R1      AB-01    False  False   True
R2      CD-02    False  False   True
R3    备注EF-03    False  False  False
R4               False   True  False
R5        NaN     True  False  False
R6      ＡＢ－０１    False  False  False
   city detail
R1   北京     朝阳
R2   上海     浦东
R3   广州    NaN
R4         NaN
R5  NaN    NaN
R6   深圳     南山
(6, 8) ['R1', 'R2', 'R3', 'R4', 'R5', 'R6']


## 本章小结

（1）str 访问器逐项处理文本。空串、缺失和不匹配是不同状态，应保留原文与必要标记。

（2）contains 查找任意位置，match 检查开头，fullmatch 检查整段。普通分隔符和标点优先明确使用字面解释。

（3）split 按分隔符拆分，extract 按捕获组提取首个匹配，extractall 展开所有匹配；要核对结果行数和来源标签。

（4）默认 str 与显式 string 的缺失语义不同。存储后端、正则模式和 Unicode 规范化都会影响结果，不能只核对表面文本。

## 练习

（1）清理下面的编号：去首尾空白、转大写、检查是否完整符合两位大写字母、连字符和两位 ASCII 数字。保留全部原行，分别标记空串、缺失和格式不合格。

In [17]:
codes = pd.Series([" xy-09 ", "XY-09备注", " ", None, "中文"], index=["A", "B", "C", "D", "E"], dtype="string")
# 在此构造保留原文和清理结果的小表。
# 检查：只有 A 格式合格；C 清理后为空串，D 缺失；总行数仍为 5。
# 提示：可空比较会传播缺失值，布尔标记需要明确填充规则。

（2）预测两种 contains 写法的结果，再运行核对。随后把任务改为“只接收整段文本恰好为 A.1”，选择合适方法并解释为什么原来的包含检查不够。

In [18]:
text = pd.Series(["A.1", "AB1", "备注A.1", "", None])
# 在此先记录两组预测。
print(text.str.contains("A.1", regex=False, na=False).tolist())
print(text.str.contains("A.1", regex=True, na=False).tolist())
# 在此完成整段字面文本检查；结果应只有第一项为 True。
# 用注释解释采用相等比较或其他写法的理由。

[True, False, True, False, False]
[True, True, True, False, False]


（3）从每条备注提取第一个区域代码与两位编号，保留所有原行及缺失结果。随后要求改为“列出一条备注中的全部编号并能追溯来源”，选择方法，解释输出行数和索引发生了什么变化。

In [19]:
notes = pd.Series(["AB-01 CD-02", "无编号", None, "EF-03"], index=["R1", "R2", "R3", "R4"], dtype="string")
# 先完成首个匹配提取；检查：4 行、2 列，R2 和 R3 对应字段缺失。
# 再完成全部匹配提取；检查：3 个匹配，能识别 R1 的两次匹配和 R4 的一次匹配。
# 在注释中说明为什么第二种结果不能直接当作完整的原记录清单。

（4）拆分下面的地址，只按首个竖线分成城市和详细地址，保留详细地址里的后续竖线。随后新增约束：“展示文本中缺失字段用‘未知’代替，但原字段仍保留缺失”，选择拼接方法并解释为何不直接覆盖原字段。

In [20]:
addresses = pd.Series(["北京|朝阳|望京", "上海", "", None], index=["A", "B", "C", "D"], dtype="string")
# 在此拆分并生成额外的展示列。
# 检查：城市和详细地址共 4 行；A 的详细地址为 朝阳|望京，B 的详细地址缺失。
# 展示列可用 cat 的 na_rep；注意空串不是缺失，不能自动变为“未知”。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [Working with text data](https://pandas.pydata.org/docs/user_guide/text.html) 的 Text data types、String methods、Splitting and replacing strings、Extracting substrings、The four StringDtype variants；[字符串类型迁移](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html) 的默认 str、PyArrow 存储与缺失语义；[StringDtype](https://pandas.pydata.org/docs/reference/api/pandas.StringDtype.html) 的 storage、na_value；[strip](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html)、[slice](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.slice.html) 的处理范围；[contains](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html)、[match](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.match.html)、[fullmatch](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html) 的匹配范围、regex、case、na；[split](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html) 的 n、expand、regex；[extract](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html) 的捕获组和形状、[extractall](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extractall.html) 的匹配行与 MultiIndex；[replace](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.replace.html) 的字面与正则替换；[cat](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.cat.html) 的标签对齐、join、na_rep；[normalize](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.normalize.html) 的规范化形式；[3.0.1 发布说明](https://pandas.pydata.org/docs/whatsnew/v3.0.1.html#fixed-regressions) 的 Arrow 字符串 RE2 模式支持。本章提取结果的 dtype 另以当前输入实际打印，不沿用部分页面中“总是 object”的旧描述。 |
| Python 官方文档（Python 3.12） | [re](https://docs.python.org/3.12/library/re.html) 的 Regular Expression Syntax：字符集合、重复次数、命名捕获组、非捕获组、Unicode 十进制数字，以及 search、match、fullmatch；[unicodedata.normalize](https://docs.python.org/3.12/library/unicodedata.html#unicodedata.normalize) 的 NFC、NFKC、规范与兼容等价；[warnings](https://docs.python.org/3.12/library/warnings.html#testing-warnings) 的局部警告捕获。 |
| Apache Arrow 官方文档（Arrow 25.0.1） | [match_substring_regex](https://arrow.apache.org/docs/python/generated/pyarrow.compute.match_substring_regex.html) 的任意位置匹配、缺失传播和大小写条件。 |
| GitHub 官方项目（版本化来源） | [Syntax](https://github.com/google/re2/wiki/Syntax) 的 Perl character classes、Grouping 与 Empty strings：ASCII 字符类及部分不支持的正则语法；用于对照 Python re，具体 pandas 方法仍按当前后端核对。 pandas v3.0.6 文档源码：[text](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/text.rst)、[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)、[v3.0.1](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/whatsnew/v3.0.1.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |